# MongoDB Implementation

**Student:** Vishal Singh  
**Database:** MongoDB Atlas  
**Dataset:** All_Beauty category from Amazon Reviews 2023

This notebook implements and benchmarks 5 queries for Task 1 using MongoDB Atlas.

### Timing Method
Query performance is measured using MongoDB's built-in `explain('executionStats')`, which returns `executionTimeMillis` — the actual server-side execution time. This is more accurate than Python's `time` library as it excludes network latency and Python overhead.

### Benchmarking Approach
Each query is run **10 times** (once per item in the shared test list). We report the **mean execution time in milliseconds**.

## 1. Imports & Connection

In [1]:
import sys
import pandas as pd
from datetime import datetime
from pymongo import MongoClient
sys.path.append('..')

# Only import what we need from helper , not importing qdrant as it throw error 
from common.helper import connect_mongodb, get_mongodb_collection_stats
from common.const import MONGODB_URI, MONGODB_DATABASE, MONGODB_PRODUCTS_COLLECTION, MONGODB_REVIEWS_COLLECTION

print("Imports done")

Imports done


In [2]:
# Connect
db,client = connect_mongodb()
products = db[MONGODB_PRODUCTS_COLLECTION]
reviews  = db[MONGODB_REVIEWS_COLLECTION]

print("Connected to MongoDB")
print(f"Database: {MONGODB_DATABASE}")
products_stats = get_mongodb_collection_stats(db, MONGODB_PRODUCTS_COLLECTION)
reviews_stats  = get_mongodb_collection_stats(db, MONGODB_REVIEWS_COLLECTION)

print(f"Products :{products_stats['count']:,}")
print(f"Reviews: {reviews_stats['count']:,}")

Connected to MongoDB
Database: amazon_reviews
Products :112,590
Reviews: 701,528


## 2. Data Structure

MongoDB stores data as BSON documents (JSON-like). Collections:

**products** — title, description, category, average_rating, price, images, store, parent_asin, etc.

**reviews** — user_id, parent_asin, rating, title, text, timestamp, verified_purchase, etc.

### Indexing Strategy
The following indexes are in place:
- `products`: single-field index on `parent_asin`
- `reviews`: single-field index on `parent_asin`
- `reviews`: single-field index on `user_id`
- `reviews`: compound index on `(parent_asin, timestamp)` — supports recent-reviews query efficiently

In [3]:
# Show sample data
sample_product = products.find_one()
print("Sample Product:")
print(f"parent_asin : {sample_product['parent_asin']}")
print(f"title: {sample_product['title']}")
print(f"average_rating : {sample_product['average_rating']}")
print(f"store: {sample_product.get('store', 'N/A')}")

print()
sample_review = reviews.find_one()
print("Sample Review:")
print(f"parent_asin : {sample_review['parent_asin']}")
print(f"user_id: {sample_review['user_id']}")
print(f"rating: {sample_review['rating']} stars")
print(f"title : {sample_review['title']}")

Sample Product:
parent_asin : B01CUPMQZE
title: Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)
average_rating : 4.8
store: Howard Products

Sample Review:
parent_asin : B00YQ6X8EO
user_id: AGKHLEW2SOWHNMFQIJGBECAF7INQ
rating: 5.0 stars
title : Such a lovely scent but not overpowering.


## 3. Timing Helper

Uses MongoDB's `explain('executionStats')` to get server-side execution time in ms.
This is the correct approach — `time.time()` measures Python+network overhead, not actual query time.

In [4]:
def measure_with_explain(collection, query_type, query_filter, projection=None, sort=None, limit=None):
    """

    Measure mongoDB query execution time using explain('executionStats').
    returns execution time in milliseconds (server-side only).
    query_type: 'find' or 'aggregate'
    """
    if query_type == 'find':
        cursor = collection.find(query_filter, projection or {})
        if sort:
            cursor = cursor.sort(*sort)
        if limit:
            cursor = cursor.limit(limit)
        explain_result = cursor.explain()
        exec_time_ms = explain_result['executionStats']['executionTimeMillis']
    
    # For aggregations, use command() directly
    elif query_type == 'aggregate':
        explain_result = collection.database.command(
            'explain',
            {'aggregate': collection.name, 'pipeline': query_filter, 'cursor': {}},
            verbosity='executionStats'
        )
        exec_time_ms = explain_result['stages'][0]['$cursor']['executionStats']['executionTimeMillis']
    return exec_time_ms

def run_10_and_average(query_func, items):
    """
    Run query function 10 times (once per item) and return average time in ms(milli second).
    query_func must return execution_time_ms.
    items: list of 10 test inputs.
    """
    times = []
    for item in items:
        t = query_func(item)
        times.append(t)
    avg = sum(times) / len(times)
    return avg, times

print("timing helpers ready")

timing helpers ready


## 4. Test Data (from Notion)

Shared test inputs so all databases are benchmarked on the same data.

In [5]:
# From Notion — shared test inputs
PRODUCT_IDS = ['B07484MS4G', 'B01FYDF3VI', 'B01C3PMM7S', 'B012HUTKTM', 'B075R6NQ1W','B08DCZGQJ5', 'B077WBYPLG', 'B001KYU8UM', 'B0919HCJDN', 'B08MQCXGJM']

KEYWORDS = [
    'Plumping Whip',
    'Eye Sticker',
    'Hair Scrunchies Colorful',
    'facial jade roller',
    'Professional Supreme Performance Braiding',
    'vintage English floral fragrance',
    'Water Weightless Hydration Conditioner',
    'hottest beauty product of 2019',
    'Mist Coolant Lubrication Spray System',
    'keep the skin calm and soothed'
]

USER_IDS = [
    'AG5JJZKUDTC5PREHPTI5NOC75M2Q', 'AFQIHDRZZ7RGRLSFKDGCYPBHHTKA','AFXCPSCWLOSVQZHN7FAECOYAT3RA', 'AHT6VS3G3JSA7KBOCBT6KPUCJX7A',
    'AH4G4SNKSIWZNGZ5EDNJTRORKIMQ', 'AG4JLUSDCYTYSSE5SQADXOPATLTA','AGXVAAK6XJ7C2GUTYQEMBOGQ7JGQ', 'AGKPEXI6SAACYVXRNMICW3NKG2XA',
    'AHA272AOVKWZUXL672G653A2MTIA', 'AH6X5VUV5AUGO3G55G7AL6GZ2QZQ'
]
print(f"Product IDs : {len(PRODUCT_IDS)}")
print(f"Keywords : {len(KEYWORDS)}")
print(f"User IDs : {len(USER_IDS)}")

Product IDs : 10
Keywords : 10
User IDs : 10


## 5. Queries

### Query 1: Product Information Retrieval
Given a product ID (`parent_asin`), retrieve the product title and full product description.

In [6]:
def query1_product_info(parent_asin):
    #Retrieve title and description for a product. Returns result and exec time in ms
    result = products.find_one(
        {'parent_asin': parent_asin},
        {'title': 1, 'description': 1, '_id': 0}
    )
    return result
def time_query1(parent_asin):
    #Returns server-side execution time in ms for Query 1.
    return measure_with_explain(
        products,
        'find',
        {'parent_asin': parent_asin},
        projection={'title': 1, 'description': 1, '_id': 0}
    )

# Show sample result
sample = query1_product_info(PRODUCT_IDS[0])
if sample:
    print(f"Product: {PRODUCT_IDS[0]}")
    print(f"Title: {sample['title']}")
    desc = sample.get('description', [])
    desc_text = ' '.join(desc) if isinstance(desc, list) else str(desc)
    print(f"Desc: {desc_text[:150]}..." if len(desc_text) > 150 else f"Desc: {desc_text}")
else:
    print(f"No result for {PRODUCT_IDS[0]}")

# Benchmark: 10 runs
avg_q1, times_q1 = run_10_and_average(time_query1, PRODUCT_IDS)
print(f"\nQuery 1 — Average: {avg_q1:.2f} ms  |  All times: {[f'{t:.1f}' for t in times_q1]}")

Product: B07484MS4G
Title: 5 x Spank Me Tattoo Lettering in Black - Sexy Kinky BDSM Fetish Tattoo (5)
Desc: 5 x "Spank Me" lettering tattoo with a size of 9.3 x 2.4 cm in black.  The tattoo is for ideal for clubs or fetish parties.  Includes german instructi...

Query 1 — Average: 0.00 ms  |  All times: ['0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0']


### Query 2: Recent Reviews
Given a product ID, retrieve the most recent 10 customer reviews (rating, title, text, timestamp).

In [7]:
def query2_recent_reviews(parent_asin, n=10):
    """
    Retrieve N most recent reviews for a product.
    """
    result = reviews.find(
        {'parent_asin': parent_asin},
        {'rating': 1, 'title': 1, 'text': 1, 'timestamp': 1, '_id': 0}
    ).sort('timestamp', -1).limit(n)
    return list(result)

def time_query2(parent_asin):
    #Returns server-side execution time in ms for Query 2
    return measure_with_explain(
        reviews,
        'find',
        {'parent_asin': parent_asin},
        projection={'rating': 1, 'title': 1, 'text': 1, 'timestamp': 1, '_id': 0},
        sort=('timestamp', -1),
        limit=10
    )

# Show sample result
sample = query2_recent_reviews(PRODUCT_IDS[0])
print(f"Product: {PRODUCT_IDS[0]}  |  Found {len(sample)} reviews")
for i, r in enumerate(sample[:3], 1):
    date = datetime.fromtimestamp(r['timestamp']/1000).strftime('%Y-%m-%d')
    print(f"{i}. {r['rating']}  [{date}]  {r['title']}")
    print(f" {r.get('text','')[:80]}...")

# Benchmark: 10 runs
avg_q2, times_q2 = run_10_and_average(time_query2, PRODUCT_IDS)
print(f"\nQuery 2 — Average: {avg_q2:.2f} ms  |  All times: {[f'{t:.1f}' for t in times_q2]}")

Product: B07484MS4G  |  Found 4 reviews
1. 5.0  [2019-12-03]  Very fun easy to install
 Very fun look great...
2. 5.0  [2019-06-14]  Cool!
 I've ordered several variants of these tattoos, and found them quite entertainin...
3. 5.0  [2018-10-30]  Fun!
 I love these temporary tattoos!  Such fun!  They last longer than some others....

Query 2 — Average: 0.00 ms  |  All times: ['0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0']


### Query 3: Keyword Search
Given a keyword, return all products whose title or description contains the keyword.

**Note:** Both the keyword and the title/description are lowercased before comparison to ensure consistent matching.

**Note on storage:** MongoDB Atlas free tier has a 512MB limit. Our dataset is ~546MB so a full-text index could not be created. Using case-insensitive regex instead — functionally equivalent.

In [8]:
def query3_keyword_search(keyword):
    """
    Search products by keyword in title or description.
    Keyword is lowercased; regex is case-insensitive (equivalent to lowercasing both sides).
    """
    keyword_lower = keyword.lower()  # it lowercase the keyword
    result = products.find(
        {
            '$or': [
                {'title': {'$regex': keyword_lower, '$options': 'i'}},  # 'i' = case-insensitive
                {'description': {'$regex': keyword_lower, '$options': 'i'}}  # treats description as lowercase too
            ]
        },
        {'title': 1, 'parent_asin': 1, '_id': 0}
    )
    return list(result)

def time_query3(keyword):
    #Returns server-side execution time in ms for Query 3.
    keyword_lower = keyword.lower()
    return measure_with_explain(
        products,
        'find',
        {
            '$or': [
                {'title': {'$regex': keyword_lower, '$options': 'i'}},
                {'description': {'$regex': keyword_lower, '$options': 'i'}}
            ]
        },
        projection={'title': 1, 'parent_asin': 1, '_id': 0}
    )

# Show sample result
sample = query3_keyword_search(KEYWORDS[0])
print(f"Keyword: '{KEYWORDS[0]}'  |  Found {len(sample)} products")
for i, p in enumerate(sample[:3], 1):
    print(f"  {i}.[{p['parent_asin']}] {p['title']}")

# benchmark: 10 runs (one per keyword)
avg_q3, times_q3 = run_10_and_average(time_query3, KEYWORDS)
print(f"\nQuery 3 — Average: {avg_q3:.2f} ms |All times: {[f'{t:.1f}' for t in times_q3]}")

Keyword: 'Plumping Whip'  |  Found 1 products
  1.[B01BW4ZU16] T&G Plumping Whip Size 2.82z Tony & Guy Volumizing Plumping Whip 2.82z

Query 3 — Average: 185.90 ms |All times: ['184.0', '266.0', '202.0', '181.0', '179.0', '161.0', '173.0', '154.0', '178.0', '181.0']


### Query 4: User Review History
Given a user ID, retrieve all products reviewed by that user with ratings and review dates.

In [9]:
def query4_user_history(user_id):
    """
    Retrieve all reviews by a user, sorted newest first
    """
    result = reviews.find(
        {'user_id': user_id},
        {'parent_asin': 1, 'rating': 1, 'timestamp': 1, '_id': 0}
    ).sort('timestamp', -1)
    return list(result)

def time_query4(user_id):
    #Returns server-side execution time in ms for Query 4
    return measure_with_explain(
        reviews,
        'find',
        {'user_id': user_id},
        projection={'parent_asin': 1, 'rating': 1, 'timestamp': 1, '_id': 0},
        sort=('timestamp', -1)
    )

# Show sample result
sample = query4_user_history(USER_IDS[0])
print(f"User: {USER_IDS[0]}")
print(f"Total reviews: {len(sample)}")
for i, r in enumerate(sample[:5], 1):
    date = datetime.fromtimestamp(r['timestamp']/1000).strftime('%Y-%m-%d')
    print(f"  {i}. Product {r['parent_asin']}  |  {r['rating']}★  |  {date}")

# Benchmark: 10 runs (one per user)
avg_q4, times_q4 = run_10_and_average(time_query4, USER_IDS)
print(f"\nQuery 4 — Average: {avg_q4:.2f} ms  |  All times: {[f'{t:.1f}' for t in times_q4]}")

User: AG5JJZKUDTC5PREHPTI5NOC75M2Q
Total reviews: 1
  1. Product B098KCD857  |  5.0★  |  2021-11-15

Query 4 — Average: 0.00 ms  |  All times: ['0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0']


### Query 5: Product Statistics
For a given product, retrieve: average rating, total reviews, and count of 1★ through 5★ reviews.

All 5 star levels are always returned explicitly, even if count = 0.

In [10]:
def query5_product_stats(parent_asin):
    """
    Retrieve aggregate stats for a product:
    1) average rating
    2) total reviews
    3) count of 1, 2, 3, 4, 5 star reviews (all explicitly, even if 0)
    """
    pipeline = [
        {'$match': {'parent_asin': parent_asin}},
        {'$group': {
            '_id': None,
            'avg_rating':   {'$avg': '$rating'},
            'total':        {'$sum': 1},
            'stars_1': {'$sum': {'$cond': [{'$eq': ['$rating', 1]}, 1, 0]}},
            'stars_2': {'$sum': {'$cond': [{'$eq': ['$rating', 2]}, 1, 0]}},
            'stars_3': {'$sum': {'$cond': [{'$eq': ['$rating', 3]}, 1, 0]}},
            'stars_4': {'$sum': {'$cond': [{'$eq': ['$rating', 4]}, 1, 0]}},
            'stars_5': {'$sum': {'$cond': [{'$eq': ['$rating', 5]}, 1, 0]}},
        }}
    ]
    result = list(reviews.aggregate(pipeline))
    if not result:
        return {'avg_rating': 0, 'total': 0, 'stars_1': 0, 'stars_2': 0, 'stars_3': 0, 'stars_4': 0, 'stars_5': 0}
    return result[0]

def time_query5(parent_asin):
    # Returns server-side execution time in ms for Query 5.
    pipeline = [
        {'$match': {'parent_asin': parent_asin}},
        {'$group': {
            '_id': None,
            'avg_rating':   {'$avg': '$rating'},
            'total':        {'$sum': 1},
            'stars_1': {'$sum': {'$cond': [{'$eq': ['$rating', 1]}, 1, 0]}},
            'stars_2': {'$sum': {'$cond': [{'$eq': ['$rating', 2]}, 1, 0]}},
            'stars_3': {'$sum': {'$cond': [{'$eq': ['$rating', 3]}, 1, 0]}},
            'stars_4': {'$sum': {'$cond': [{'$eq': ['$rating', 4]}, 1, 0]}},
            'stars_5': {'$sum': {'$cond': [{'$eq': ['$rating', 5]}, 1, 0]}},
        }}
    ]
    explain_result = reviews.database.command(
        'explain',
        {'aggregate': reviews.name, 'pipeline': pipeline, 'cursor': {}},
        verbosity='executionStats'
    )
    try:
        exec_time_ms = explain_result['stages'][0]['$cursor']['executionStats']['executionTimeMillis']
    except (KeyError, IndexError):
        # Fallback path for different MongoDB explain output formats
        exec_time_ms = explain_result.get('executionStats', {}).get('executionTimeMillis', 0)
    return exec_time_ms

# Show sample result
sample = query5_product_stats(PRODUCT_IDS[0])
print(f"Product : {PRODUCT_IDS[0]}")
print(f"Average rating : {sample['avg_rating']:.2f}"  if sample['avg_rating'] else "Average rating : N/A")
print(f"Total reviews  : {sample['total']}")
print(f"1star reviews : {sample['stars_1']}")
print(f"2star reviews : {sample['stars_2']}")
print(f"3star reviews : {sample['stars_3']}")
print(f"4star reviews : {sample['stars_4']}")
print(f"5star reviews : {sample['stars_5']}")

# Benchmark: 10 runs
avg_q5, times_q5 = run_10_and_average(time_query5, PRODUCT_IDS)
print(f"\nQuery 5 — Average: {avg_q5:.2f} ms |All times: {[f'{t:.1f}' for t in times_q5]}")

Product : B07484MS4G
Average rating : 4.00
Total reviews  : 4
1star reviews : 1
2star reviews : 0
3star reviews : 0
4star reviews : 0
5star reviews : 3

Query 5 — Average: 1.00 ms |All times: ['1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0']


## 6. Performance Summary

Mean execution time across 10 runs per query (server-side ms via `explain`).

In [11]:
summary = [
    {'Query': '1. Product Info Retrieval','Mean (ms)': round(avg_q1, 2), 'All runs (ms)': [round(t,1) for t in times_q1]},
    {'Query': '2. Recent Reviews (N=10)', 'Mean (ms)': round(avg_q2, 2), 'All runs (ms)': [round(t,1) for t in times_q2]},
    {'Query': '3. Keyword Search',  'Mean (ms)': round(avg_q3, 2), 'All runs (ms)': [round(t,1) for t in times_q3]},
    {'Query': '4. User Review History', 'Mean (ms)': round(avg_q4, 2), 'All runs (ms)': [round(t,1) for t in times_q4]},
    {'Query': '5. Product Statistics', 'Mean (ms)': round(avg_q5, 2), 'All runs (ms)': [round(t,1) for t in times_q5]},
]

df = pd.DataFrame(summary)[['Query', 'Mean (ms)']]
print("MongoDB query performance — All_Beauty dataset")
print(f"Timing: MongoDB explain('executionStats') |Runs: 10 per query\n")
print(df.to_string(index=False))

print(f"\nDetailed run times:")
for row in summary:
    print(f"  {row['Query']}: {row['All runs (ms)']}")

MongoDB query performance — All_Beauty dataset
Timing: MongoDB explain('executionStats') |Runs: 10 per query

                    Query  Mean (ms)
1. Product Info Retrieval        0.0
 2. Recent Reviews (N=10)        0.0
        3. Keyword Search      185.9
   4. User Review History        0.0
    5. Product Statistics        1.0

Detailed run times:
  1. Product Info Retrieval: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  2. Recent Reviews (N=10): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  3. Keyword Search: [184, 266, 202, 181, 179, 161, 173, 154, 178, 181]
  4. User Review History: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  5. Product Statistics: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 7. Results & Analysis

### Timing Method
All times use MongoDB's `explain('executionStats').executionTimeMillis` — this is pure server-side query execution time, excluding Python overhead and network round-trip.

### Indexing Strategy
- **products.parent_asin** — single-field index. Enables O(log n) lookups for Queries 1 and 5.
- **reviews.parent_asin** — single-field index. Used by Queries 2 and 5 to filter by product.
- **reviews.(parent_asin, timestamp)** — compound index. Supports Query 2's sort-by-recency without a full collection scan.
- **reviews.user_id** — single-field index. Used by Query 4 to retrieve user history efficiently.

### What worked well
- Point lookups (Q1, Q2, Q5) are fast thanks to indexes on `parent_asin`
- MongoDB's aggregation pipeline (Q5) handles conditional counting per star level neatly in a single pass
- Document model suits the Amazon data — products have variable fields (some have videos, some don't) that fit naturally without null columns

### Known Limitations
- **Q3 (Keyword Search):** MongoDB Atlas free tier has a 512MB storage limit. Our dataset is ~546MB so a proper full-text search index could not be created. Using case-insensitive regex instead — functionally correct but slower than a text index would be. On a paid tier this would use `$text` with a text index for significantly better performance.
- **Q4 (User History):** Some users have very few reviews in the Beauty category, making results sparse for certain user IDs in the test list.

### MongoDB vs SQL for this use case
MongoDB is well-suited here because the product data is semi-structured (variable fields per product), and the document model avoids the need for joins between products and their metadata. SQL databases like Supabase/PostgreSQL offer better performance for complex analytical queries but require a rigid schema. For a production e-commerce platform, MongoDB would be recommended for the product catalogue, while a relational DB would handle transactional and reporting workloads.

In [12]:
client.close()
print("Connection closed.")

Connection closed.
